# Признаки, переобучение и регуляризация

CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

- Отличать признак, доступный до прогноза, от ответа и сведений из будущего.
- Узнавать переобучение по ошибкам на обучающих и новых заказах.
- Проверять, помогает ли ограничение сложности дерева на новых заказах.

## План занятия

1. Добавим к знакомым признакам оценку объёма пластика.
2. Сравним деревья по ошибке на обучении и на новых заказах.
3. Ограничим число листьев косвенно и проверим результат в практике.

## Ещё одно сведение о фигурке

В [занятии 1.4](../04-decision-trees/README.md) дерево предсказывало время 3D-печати по высоте фигурки и числу цветов. В [занятии 1.1](../01-first-experiment/README.md) мы видели, что фигуры с одинаковыми значениями этих двух признаков могут печататься разное время. Теперь мастерская стала получать из проекта ещё одно число **до запуска принтера**: оценку объёма пластика в кубических сантиметрах. Это третий признак, `plastic_cm3`. Его можно передать модели вместе с `height_cm` и `colors`.

В новом учебном наборе 300 выполненных заказов для обучения и 100 новых для проверки. Все данные **синтетические**: генератор и фиксированные начальные значения случайных чисел лежат в [data/generate.py](data/generate.py). Правильное время зависит от объёма, числа смен цвета и не показанных модели задержек. Это не формула настоящего принтера. В [train.csv](data/train.csv) ответы известны. В [new.csv](data/new.csv) столбец `minutes` заполнен знаками `?`; правильные значения хранятся отдельно в [new_answers.csv](data/new_answers.csv).

Например, у заказов `tr0055` и `tr0142` одинаковая высота **3 см** и **два цвета**. Но оценка объёма равна **3** и **10 см³**, а печать заняла **12** и **22 минуты**. Только по первым двум признакам модель не различит эти строки. Объём помогает, хотя сам по себе не объясняет всё: возможны скрытые задержки.

Полезный признак должен отвечать на два вопроса: **доступен ли он в момент прогноза** и **уменьшает ли он ошибку на новых заказах**? Номер заказа может случайно совпасть с ответом в учебной таблице, но не описывает работу принтера. Фактическое время печати стало известно лишь после печати: подставить его в признаки означало бы раскрыть ответ модели, то есть допустить **утечку данных**.

## Один и тот же способ проверки

Как и прежде, измеряем **MAE**, средний модуль ошибки в минутах. Модель обучается только на 300 заказах. Новые 100 заказов мы используем для сравнения заранее выбранных вариантов; их ответы не передаём в `fit`. Такое сравнение помогает выбирать настройки, поэтому после него эти 100 заказов уже нельзя называть нетронутым итоговым тестом. Честную итоговую оценку подробно разберём в занятии 1.8.

Сначала фиксируем четыре варианта знакомого `DecisionTreeRegressor`:

| Вариант | Признаки | Ограничение |
| --- | --- | --- |
| A | высота, число цветов | глубина не больше 4 |
| B | те же и объём пластика | глубина не больше 4 |
| C | те же три | ограничений нет |
| D | те же три | в каждом листе не меньше 5 обучающих заказов |

В A и B изменяется только набор признаков: это сравнение показывает вклад объёма **при данной глубине**. В B и C меняется только ограничение дерева. В C и D меняется минимальное число обучающих заказов в листе. По одному такому сравнению нельзя объявить признак или настройку лучшими для любой задачи.

In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

train = pd.read_csv("data/train.csv")
new = pd.read_csv("data/new.csv")
basic = ["height_cm", "colors"]
extended = basic + ["plastic_cm3"]

variants = {
    "A: два признака, глубина 4": (basic, 4, 1),
    "B: три признака, глубина 4": (extended, 4, 1),
    "C: три признака, без ограничения": (extended, None, 1),
    "D: три признака, от 5 в листе": (extended, None, 5),
}
models = {}
for name, (columns, depth, leaf) in variants.items():
    model = DecisionTreeRegressor(
        max_depth=depth, min_samples_leaf=leaf, random_state=0
    )
    model.fit(train[columns], train["minutes"])
    models[name] = (model, columns)


Теперь правила зафиксированы. Открываем ответы, сопоставляем их по `order_id` и вычисляем ошибки. Проверка `one_to_one` не даст незаметно продублировать заказ при соединении таблиц.

In [ ]:
answers = pd.read_csv("data/new_answers.csv")
checked = new.merge(
    answers, on="order_id", validate="one_to_one",
    suffixes=("_hidden", "")
)
assert len(checked) == len(new) == 100

for name, (model, columns) in models.items():
    train_mae = mean_absolute_error(
        train["minutes"], model.predict(train[columns])
    )
    new_mae = mean_absolute_error(
        checked["minutes"], model.predict(checked[columns])
    )
    print(name, round(train_mae, 2), round(new_mae, 2), model.get_n_leaves())


| Вариант | MAE на обучении, мин | MAE на новых заказах, мин | Листьев |
| --- | ---: | ---: | ---: |
| A: два признака, глубина 4 | 3,21 | 3,83 | 16 |
| B: три признака, глубина 4 | 2,39 | 3,38 | 16 |
| C: три признака, без ограничения | 0,90 | 3,62 | 183 |
| D: три признака, от 5 в листе | 2,06 | 3,18 | 46 |



[Код построения диаграммы](plot_errors.py) вычисляет эти же ошибки из CSV и сохраняет SVG. Высота оранжевого столбика показывает ошибку на новых заказах; её измеряем в **минутах**, как и время печати.

## Почему маленькая ошибка на обучении обманчива

Сравним B и C. Без ограничения дерево создало **183 листа** для 300 обучающих заказов. Оно снизило обучающую MAE с **2,39** до **0,90** минуты: многие листья отвечают по одному или нескольким уже увиденным заказам. На новых заказах MAE, напротив, выросла с **3,38** до **3,62** минуты. Модель уловила подробности прошлого набора, которые плохо переносятся на новые фигурки. Такое расхождение называют **переобучением**.

Это не означает, что глубокое дерево всегда хуже. Здесь мы наблюдаем результат **конкретного разделения и конкретных синтетических данных**. Проверять нужно оба числа: ошибка обучения показывает, насколько модель подстроилась под известные строки, а ошибка новых заказов — как она справилась с этой проверкой.

## Ограничим сложность

**Регуляризация** — общее название способов сдержать гибкость модели, чтобы ей было труднее запоминать случайные детали обучения. Деревья тоже можно регуляризовать: ограничить глубину (`max_depth`), потребовать минимум обучающих заказов в листе (`min_samples_leaf`) или ограничить число листьев (`max_leaf_nodes`). Здесь проверяем `min_samples_leaf=5`: в каждом конечном листе должно быть не меньше пяти обучающих заказов. Эти ограничения влияют на построение дерева. В линейной регрессии, которую отдельно разберём в занятии 1.3, для той же цели можно добавить к функции потерь штраф за большие коэффициенты. Штраф не вычитается из готового прогноза.

Вариант D с новым признаком имеет **46 листьев**, обучающую MAE **2,06** и MAE на новых заказах **3,18** минуты. По сравнению с C он хуже подстроился под обучение (**2,06 вместо 0,90**), зато лучше справился с новыми заказами (**3,18 вместо 3,62**). Здесь ограничение помогло. Если сделать листья слишком крупными, модель может перестать замечать полезные различия: это проверим сами.

Добавление признака и регуляризация решают разные вопросы. Сравнение A → B показывает, что объём помог дереву той же глубины. Сравнение C → D показывает, что ограничение гибкости помогло дереву с одинаковыми тремя признаками. Сравнивать варианты следует по одной и той же метрике и одним и тем же новым заказам.

## Практика

1. Найдите в обучающей таблице `tr0055` и `tr0142`. Объясните, что дерево может узнать об их различии из первых двух признаков и что добавляет третий. Не называйте объём правильным временем печати.
2. Запустите листинги или [ноутбук](notebook.ipynb) рядом с папкой `data`. Воспроизведите четыре строки таблицы ошибок. Убедитесь, что `new_answers.csv` нигде не передаётся в `fit`.
3. У дерева с тремя признаками и без ограничения глубины попробуйте `min_samples_leaf=2`, `5`, `10` и `20`. Сравните ошибки и число листьев. Какое значение дало меньшую MAE на этих новых заказах? Почему нельзя сообщать это число как независимую итоговую оценку выбранной настройки?

**Самопроверка.** У `tr0055` и `tr0142` совпадают высота и число цветов, но объём равен 3 и 10 см³. Для `min_samples_leaf=2, 5, 10, 20` MAE на новых заказах составляет соответственно **3,50; 3,18; 3,26; 3,82** минуты, а число листьев — **112; 46; 21; 10**. На этом наборе лучше 5. Мы выбрали значение по тем же 100 ответам, поэтому для итоговой оценки понадобится отдельный, ещё не использованный набор.

## Типичные ошибки

- Добавлять фактическое время печати как признак: в момент прогноза оно неизвестно.
- Сравнивать разные модели только по обучающей MAE: она может снижаться одновременно с ухудшением прогноза новых заказов.
- Называть любой новый столбец полезным, не проверив, что он доступен до печати и помогает на новых данных.
- Подбирать настройку по 100 новым ответам, а затем представлять ошибку на них как независимый итоговый тест.

## Самопроверка

### 1. Что в этой задаче можно узнать до начала печати?

A. Фактическое время окончания заказа.<br>
B. Объём пластика, рассчитанный из проекта.<br>
C. Ошибку модели на этом заказе.<br>
D. Ответ из `new_answers.csv`.

<details><summary>Подсказка</summary>

Вспомните, какие данные есть в проекте фигурки до запуска принтера.

</details>


### 2. Почему `tr0055` и `tr0142` не различаются по двум старым признакам?

A. У них одинаковые высота и число цветов.<br>
B. У них одинаковое время печати.<br>
C. У них одинаковый объём пластика.<br>
D. Они находятся в проверочной части.

<details><summary>Подсказка</summary>

Сравните значения `height_cm` и `colors` у двух строк.

</details>


### 3. Что показывает сравнение A и B?

A. Пользу нового признака при одинаковой максимальной глубине.<br>
B. Влияние другого разбиения данных.<br>
C. Влияние числа заказов в листе.<br>
D. Точную скорость настоящего принтера.

<details><summary>Подсказка</summary>

Найдите единственное различие между A и B в таблице вариантов.

</details>


### 4. Как изменилась MAE при переходе от B к C?

A. Уменьшилась и на обучении, и на новых заказах.<br>
B. Выросла и на обучении, и на новых заказах.<br>
C. Уменьшилась на обучении, выросла на новых заказах.<br>
D. Не изменилась.

<details><summary>Подсказка</summary>

Сравните оба столбца ошибок, а не только первый.

</details>


### 5. Почему 183 листа могут быть опасны при 300 обучающих заказах?

A. В некоторых листьях остаётся очень мало примеров, и дерево может подстроиться под их случайные особенности.<br>
B. Дерево перестаёт предсказывать число.<br>
C. Любое дерево с более чем десятью листьями математически неверно.<br>
D. Число цветов исчезает из таблицы.

<details><summary>Подсказка</summary>

Представьте, сколько обучающих заказов может оказаться в одном листе.

</details>


### 6. Что задаёт `min_samples_leaf=5`?

A. Не больше пяти вопросов от корня до листа.<br>
B. Не меньше пяти обучающих заказов в каждом листе.<br>
C. Ровно пять признаков.<br>
D. Не больше пяти минут ошибки.

<details><summary>Подсказка</summary>

Параметр относится к числу объектов в конечной группе.

</details>


### 7. Что произошло при переходе от C к D?

A. Обучающая MAE выросла, а MAE новых заказов снизилась.<br>
B. Обе ошибки стали равны нулю.<br>
C. Новые ответы передали в `fit`.<br>
D. Удалили объём пластика из признаков.

<details><summary>Подсказка</summary>

Сравните 0,90 и 2,06 на обучении, затем 3,62 и 3,18 на новых заказах.

</details>


### 8. Почему `order_id` не стоит брать как содержательный признак?

A. Это метка для сопоставления строк, а не свойство печати; случайная связь с ответом может не повториться.<br>
B. Номера нельзя хранить в CSV.<br>
C. Деревья умеют работать только с двумя признаками.<br>
D. Номер всегда полностью раскрывает ответ.

<details><summary>Подсказка</summary>

Подумайте, будет ли номер заказа объяснять время следующей фигурки.

</details>


### 9. Почему нельзя назвать MAE 3,18 минуты независимой итоговой оценкой варианта D после практики?

A. Потому что MAE измеряется не в минутах.<br>
B. Потому что эти же 100 ответов использовали при сравнении настроек.<br>
C. Потому что у дерева 46 листьев.<br>
D. Потому что обучение состояло из 300 строк.

<details><summary>Подсказка</summary>

Вспомните, по каким ответам выбирали значение `min_samples_leaf`.

</details>


### 10. Какой вывод разрешают данные этого занятия?

A. Объём всегда улучшает любую модель на настоящем принтере.<br>
B. Дерево без ограничений всегда ошибается больше ограниченного.<br>
C. В этом синтетическом опыте объём помог дереву глубины 4, а ограничение листа улучшило проверку гибкого дерева.<br>
D. Если ошибка обучения мала, новые заказы можно не проверять.

<details><summary>Подсказка</summary>

Ищите вывод, ограниченный именно проведённым опытом.

</details>



## Решения задач

### Задача 1

**B.** Оценка объёма получается из проекта заранее. Остальное станет известно после печати или проверки.

### Задача 2

**A.** У обоих высота 3 см и два цвета. Объём и фактическое время различаются.

### Задача 3

**A.** Обоим деревьям задана глубина 4; у B дополнительно есть объём пластика. На этих новых заказах MAE снизилась с 3,83 до 3,38 минуты.

### Задача 4

**C.** Обучающая MAE снизилась с 2,39 до 0,90, а ошибка новых заказов выросла с 3,38 до 3,62 минуты. Это признак переобучения в нашем опыте.

### Задача 5

**A.** Мелкие листья позволяют почти запомнить обучение; это может ухудшить прогноз новых заказов. Само число 183 не является универсальной границей.

### Задача 6

**B.** Это минимальное количество обучающих заказов в листе. Глубину ограничивает другой параметр — `max_depth`.

### Задача 7

**A.** Ограничение уменьшило подстройку под обучение и улучшило результат этой проверки. Оно не обещает улучшения на любом наборе.

### Задача 8

**A.** Номер помогает соединить таблицы. Он не описывает геометрию или настройки печати и может дать ложную закономерность.

### Задача 9

**B.** После выбора по проверочным ответам они уже повлияли на решение. Для независимой итоговой оценки нужен другой набор.

### Задача 10

**C.** Оба сравнения подтверждены числами на этих данных. Они не доказывают, что то же случится на каждом другом наборе.